# Baselines on the validation terms (2016–2020)

Every model has to beat these two. Each run is appended to `results/baseline.csv`.

In [1]:
import sys
from pathlib import Path

# make the repo root importable so `src` works from notebooks/micah/
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from src.config import FEATURE_COLUMNS
from src.data import add_circuit_features, load_data
from src.evaluate import RESULTS_DIR, always_petitioner, evaluate, previous_terms_majority
from src.features import add_features
from src.split import split_by_term

In [2]:
df = load_data(REPO_ROOT / "data" / "SCDB_2026_01_caseCentered_Citation.csv")
df = add_circuit_features(add_features(df))
X_train, y_train, X_val, y_val, X_test, y_test = split_by_term(df, FEATURE_COLUMNS)

Number of rows loaded: 9,409
Number of rows after decisionType filter: 8,216
Number of rows after partyWinning filter: 8,211
train: 7,611 cases, terms 1946-2015, petitioner win rate 62.8%
  val:   305 cases, terms 2016-2020, petitioner win rate 67.2%
 test:   295 cases, terms 2021-2025, petitioner win rate 69.2%


In [3]:
# Rolling baseline: each validation term is predicted from the 10 terms before it.
# Pass the full filtered df so 2017 can look back at 2016, etc.
majority_pred = previous_terms_majority(df, X_val["term"], window=10)
print("prediction per term:", majority_pred.groupby(X_val["term"]).first().to_dict())

# The baselines give only 0/1 answers, so those answers double as the score (ROC AUC 0.5 for a constant).
always_pred = always_petitioner(X_val)
evaluate("always petitioner", {}, y_val, always_pred, always_pred, owner="baseline", split="val")
evaluate("previous 10 terms majority", {"window": 10}, y_val, majority_pred, majority_pred,
         owner="baseline", split="val")

pd.read_csv(RESULTS_DIR / "baseline.csv")

prediction per term: {2016: 1, 2017: 1, 2018: 1, 2019: 1, 2020: 1}


,timestamp,owner,model,params,split,n,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp
0,2026-11-18T15:54:59,baseline,always petitioner,{},val,305,0.672131,0.5,0.672131,1.0,0.803922,0.5,0,100,0,205
1,2026-11-18T15:54:59,baseline,previous 10 terms majority,"{""window"": 10}",val,305,0.672131,0.5,0.672131,1.0,0.803922,0.5,0,100,0,205
2,2026-11-18T16:11:49,baseline,always petitioner,{},val,305,0.672131,0.5,0.672131,1.0,0.803922,0.5,0,100,0,205
3,2026-11-18T16:11:49,baseline,previous 10 terms majority,"{""window"": 10}",val,305,0.672131,0.5,0.672131,1.0,0.803922,0.5,0,100,0,205
